# CoTyle Dictionary-Controlled Texture Autoencoder — Colab Pro A100 Training

**Runtime Setup:** In Google Colab, go to **Runtime → Change runtime type → Hardware accelerator: GPU → GPU type: A100**.

In [ ]:
# 1. Verify A100 GPU & VRAM
!nvidia-smi

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone Repository (or pull latest if already cloned) & Install Dependencies
import os

REPO_URL = "https://github.com/rommaltser/codebook_encoder_decoder.git"
REPO_DIR = "/content/codebook_encoder_decoder"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# 4. Stage DTD Images to Local Colab NVMe SSD (/content/dtd/images) for 10x Faster Data Loading
# Reading 5,640 small JPGs from /content/ local SSD is ~15x faster than reading over Google Drive FUSE.
import os, glob, shutil

DRIVE_DATASET_PATH = "/content/drive/MyDrive/tau material/datasets/dtd-pseudo-labled-Qwen2.5-VL-32B-AWQ"
LOCAL_DTD_IMAGES = "/content/dtd/images"

assert os.path.exists(DRIVE_DATASET_PATH), f"Path not found in Drive: {DRIVE_DATASET_PATH}"
print("Found Drive dataset folder contents:", os.listdir(DRIVE_DATASET_PATH))

if not os.path.exists(LOCAL_DTD_IMAGES):
    os.makedirs("/content/dtd", exist_ok=True)
    drive_imgs_candidate = os.path.join(DRIVE_DATASET_PATH, "images")
    if os.path.isdir(drive_imgs_candidate):
        print("Copying images from Google Drive to local SSD /content/dtd/images ...")
        shutil.copytree(drive_imgs_candidate, LOCAL_DTD_IMAGES)
    else:
        print("Downloading official DTD dataset (dtd-r1.0.1.tar.gz) to local SSD /content/dtd ...")
        !wget -q https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz -O /content/dtd.tar.gz
        !tar -xzf /content/dtd.tar.gz -C /content/
        !rm -f /content/dtd.tar.gz

print("Total DTD categories on local SSD:", len(os.listdir(LOCAL_DTD_IMAGES)))

In [ ]:
# 5. Launch Two-Phase Training on A100 (Automatically Resumes from Phase 1 / Latest Checkpoint!)
# Set --num-clusters 6 (structural/surface only) or --num-clusters 8 (including chromatic_quality & hue)
import os

CKPT_DIR = "/content/drive/MyDrive/tau material/datasets/dtd-pseudo-labled-Qwen2.5-VL-32B-AWQ/checkpoints_cotyle"
LATEST_CKPT = os.path.join(CKPT_DIR, "checkpoint_latest.pt")
PHASE1_CKPT = os.path.join(CKPT_DIR, "checkpoint_phase1_supervisor.pt")

resume_arg = ""
if os.path.exists(LATEST_CKPT):
    print(f"Resuming from latest Phase 2 checkpoint: {LATEST_CKPT}")
    resume_arg = f'--resume "{LATEST_CKPT}"'
elif os.path.exists(PHASE1_CKPT):
    print(f"Found completed Phase 1 Supervisor checkpoint (96% acc)! Starting directly at Phase 2 (Epoch 16): {PHASE1_CKPT}")
    resume_arg = f'--resume "{PHASE1_CKPT}"'

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 train.py \
  --config configs/colab_a100.yaml \
  --img-dir "/content/dtd/images" \
  --labels-json "/content/drive/MyDrive/tau material/datasets/dtd-pseudo-labled-Qwen2.5-VL-32B-AWQ" \
  --num-clusters 6 \
  --batch-size 32 \
  {resume_arg}

In [ ]:
# 6. Resume Training (If a Colab Session Disconnects)
# Simply re-run cells 1-4 and run this cell:
CKPT_PATH = "/content/drive/MyDrive/tau material/datasets/dtd-pseudo-labled-Qwen2.5-VL-32B-AWQ/checkpoints_cotyle/checkpoint_latest.pt"
if os.path.exists(CKPT_PATH):
    !python3 train.py \
      --config configs/colab_a100.yaml \
      --resume "{CKPT_PATH}"

In [ ]:
# 7. Test Single-Attribute Dictionary Swap & Visualize Inline
from IPython.display import Image as IPyImage, display

test_img = "/content/dtd/images/bubbly/bubbly_0038.jpg"  # Also auto-resolves if you pass "bubbly_0038.png"
ckpt_path = "/content/drive/MyDrive/tau material/datasets/dtd-pseudo-labled-Qwen2.5-VL-32B-AWQ/checkpoints_cotyle/checkpoint_epoch_070.pt"

!python3 test_edit.py \
  --config configs/colab_a100.yaml \
  --checkpoint "{ckpt_path}" \
  --image "{test_img}" \
  --cluster element_shape \
  --word striped \
  --output /content/edited_comparison.png

display(IPyImage("/content/edited_comparison.png"))